In [1]:
import pandas as pd
import numpy as np
import re
import os
import logging
from datetime import datetime

In [6]:
 logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("cleaning_report.log", mode="w")
    ]
)
log = logging.getLogger(__name__)
 
INPUT_FILE  = r"C:\Users\Access\Downloads\New folder\Dataset for Data Analytics.xlsx"
OUT_CSV     =  r"C:\Users\Access\Downloads\New folder\orders_cleaned.csv"
OUT_XLSX    = "/mnt/user-data/outputs/orders_cleaned.xlsx"
REPORT_FILE = "/mnt/user-data/outputs/cleaning_report.txt"

In [7]:
log.info("Loading dataset …")
try:
    # dtype=str for ID cols prevents silent numeric coercion
    df = pd.read_excel(
        INPUT_FILE,
        sheet_name=0,
        dtype={"OrderID": str, "CustomerID": str, "TrackingNumber": str}
    )
    log.info(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} columns")
except Exception as e:
    log.error(f"Failed to load file: {e}")
    raise
 
# Snapshot original shape for the final report
original_shape = df.shape
issues = {}          

2026-05-23 10:13:44,748 [INFO] Loading dataset …
2026-05-23 10:13:45,062 [INFO] Loaded 1,200 rows × 14 columns


In [8]:
log.info("Normalising column names …")
 
def clean_col_name(name: str) -> str:
    """Convert any column name to SQL-safe snake_case."""
    name = str(name).strip()
    name = re.sub(r"[^\w\s]", "", name)          # strip special chars
    name = re.sub(r"\s+", "_", name)             # spaces → underscore
    name = name.lower()
    # prefix columns that start with a digit (SQL illegal)
    if re.match(r"^\d", name):
        name = "col_" + name
    return name
 
original_cols = list(df.columns)
df.columns    = [clean_col_name(c) for c in df.columns]
col_map       = dict(zip(original_cols, df.columns))
 
issues["column_rename"] = {
    "renamed": {k: v for k, v in col_map.items() if k != v},
    "action": "snake_case, special-chars removed"
}
log.info(f"Column rename map: {issues['column_rename']['renamed']}")



2026-05-23 10:14:25,953 [INFO] Normalising column names …
2026-05-23 10:14:25,959 [INFO] Column rename map: {'OrderID': 'orderid', 'Date': 'date', 'CustomerID': 'customerid', 'Product': 'product', 'Quantity': 'quantity', 'UnitPrice': 'unitprice', 'ShippingAddress': 'shippingaddress', 'PaymentMethod': 'paymentmethod', 'OrderStatus': 'orderstatus', 'TrackingNumber': 'trackingnumber', 'ItemsInCart': 'itemsincart', 'CouponCode': 'couponcode', 'ReferralSource': 'referralsource', 'TotalPrice': 'totalprice'}


In [9]:
log.info("Checking for duplicates …")
 
n_full_dupes   = df.duplicated().sum()
n_orderid_dupes = df.duplicated(subset=["orderid"]).sum()
 
if n_full_dupes:
    df.drop_duplicates(inplace=True)
    log.warning(f"Removed {n_full_dupes} fully duplicate rows")
 
if n_orderid_dupes:
    # Keep the last record (latest update wins)
    df.drop_duplicates(subset=["orderid"], keep="last", inplace=True)
    log.warning(f"Removed {n_orderid_dupes} duplicate OrderID rows (kept last)")
 
issues["duplicates"] = {
    "full_duplicate_rows": int(n_full_dupes),
    "duplicate_orderids":  int(n_orderid_dupes),
    "action": "Removed – last record kept for OrderID dupes"
}

2026-05-23 10:14:51,163 [INFO] Checking for duplicates …


In [10]:
log.info("Cleaning string columns …")
 
str_cols = df.select_dtypes(include="object").columns.tolist()
 
def clean_string_col(series: pd.Series) -> pd.Series:
    """Strip, normalise whitespace, remove control chars from a string column."""
    s = series.copy()
    s = s.astype(str)
    # Replace literal 'nan'/'NaN'/'None' strings with actual NaN
    s = s.replace({"nan": np.nan, "NaN": np.nan, "None": np.nan, "none": np.nan, "": np.nan})
    # Remove non-printable ASCII control characters (except tab/newline handled below)
    s = s.str.replace(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", regex=True)
    # Replace tabs / newlines inside values with a space
    s = s.str.replace(r"[\t\n\r]", " ", regex=True)
    # Normalise multiple internal spaces → single space
    s = s.str.replace(r" {2,}", " ", regex=True)
    # Strip leading / trailing whitespace
    s = s.str.strip()
    # Re-evaluate empty strings as NaN
    s = s.replace("", np.nan)
    return s
 
whitespace_issues = {}
for col in str_cols:
    original = df[col].copy()
    df[col]  = clean_string_col(df[col])
    changed  = (original != df[col]).sum()
    if changed:
        whitespace_issues[col] = int(changed)
 
issues["string_cleaning"] = {
    "columns_with_changes": whitespace_issues,
    "action": "strip, normalise whitespace, remove control chars"
}
 
# ─────────────────────────────────────────────────────────────────────────────
# 5. CATEGORICAL COLUMN STANDARDISATION
#    SQL Server lookups will fail on mixed-case / extra-space variants.
# ─────────────────────────────────────────────────────────────────────────────
log.info("Standardising categorical columns …")
 
# Define expected canonical values for each categorical column
categorical_standards = {
    "product":         ["Chair", "Desk", "Laptop", "Monitor", "Phone", "Printer", "Tablet"],
    "paymentmethod":   ["Cash", "Credit Card", "Debit Card", "Gift Card", "Online"],
    "orderstatus":     ["Cancelled", "Delivered", "Pending", "Returned", "Shipped"],
    "referralsource":  ["Email", "Facebook", "Google", "Instagram", "Referral"],
    "couponcode":      ["FREESHIP", "SAVE10", "WINTER15"],
}
 
cat_issues = {}
for col, valid_values in categorical_standards.items():
    if col not in df.columns:
        continue
 
    # Title-case normalisation (handles 'LAPTOP' → 'Laptop', etc.)
    if col == "couponcode":
        df[col] = df[col].str.upper()
    else:
        df[col] = df[col].str.title()
 
    # Detect any values outside the canonical list
    invalid_mask = df[col].notna() & ~df[col].isin(valid_values)
    n_invalid = invalid_mask.sum()
    if n_invalid:
        cat_issues[col] = {
            "invalid_values": df.loc[invalid_mask, col].value_counts().to_dict(),
            "action": "Flagged – set to NaN (review manually)"
        }
        df.loc[invalid_mask, col] = np.nan   # isolate unknowns
        log.warning(f"  {col}: {n_invalid} invalid category values → set to NaN")
 
issues["categorical_standardisation"] = cat_issues if cat_issues else "All values within expected sets"
 
# ─────────────────────────────────────────────────────────────────────────────
# 6. DATE COLUMN CLEANING & STANDARDISATION
#    SQL Server DATETIME / DATE columns require ISO 8601 (YYYY-MM-DD).
# ─────────────────────────────────────────────────────────────────────────────
log.info("Cleaning date columns …")
 
date_cols = ["date"]
date_issues = {}
 
for col in date_cols:
    if col not in df.columns:
        continue
 
    original_dtype = df[col].dtype
    df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=False)
 
    n_null_dates  = df[col].isna().sum()
    future_dates  = (df[col] > pd.Timestamp.today()).sum()
    ancient_dates = (df[col] < pd.Timestamp("2000-01-01")).sum()
 
    date_issues[col] = {
        "dtype_before":  str(original_dtype),
        "null_after_parse": int(n_null_dates),
        "future_dates":  int(future_dates),
        "dates_before_2000": int(ancient_dates),
    }
 
    # Standardise to SQL Server-compatible DATE string (YYYY-MM-DD)
    df[col] = df[col].dt.strftime("%Y-%m-%d")
 
    if n_null_dates:
        log.warning(f"  {col}: {n_null_dates} dates could not be parsed → NaT/NULL")
    if future_dates:
        log.warning(f"  {col}: {future_dates} future dates detected (OK for orders)")
 
issues["date_columns"] = date_issues
 
# ─────────────────────────────────────────────────────────────────────────────
# 7. NUMERIC COLUMN VALIDATION & OUTLIER HANDLING
# ─────────────────────────────────────────────────────────────────────────────
log.info("Validating numeric columns …")
 
numeric_rules = {
    # col_name : (min_allowed, max_allowed, nullable)
    "quantity":    (1,    999,   False),
    "unitprice":   (0.01, 99999, False),
    "itemsincart": (1,    999,   False),
    "totalprice":  (0.01, 99999, False),
}
 
numeric_issues = {}
 
for col, (min_val, max_val, nullable) in numeric_rules.items():
    if col not in df.columns:
        continue
 
    # Coerce to numeric – anything non-numeric becomes NaN
    df[col] = pd.to_numeric(df[col], errors="coerce")
 
    n_null = df[col].isna().sum()
    n_negative = (df[col] < 0).sum()
    n_zero = (df[col] == 0).sum()
 
    # IQR-based outlier detection (flag only – NOT removed)
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_fence = Q1 - 1.5 * IQR
    upper_fence = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower_fence) | (df[col] > upper_fence)).sum()
 
    # Clamp values outside absolute business rules
    n_below = (df[col] < min_val).sum()
    n_above = (df[col] > max_val).sum()
 
    if n_below:
        df.loc[df[col] < min_val, col] = np.nan
        log.warning(f"  {col}: {n_below} values below {min_val} → set to NaN")
    if n_above:
        df.loc[df[col] > max_val, col] = np.nan
        log.warning(f"  {col}: {n_above} values above {max_val} → set to NaN")
 
    # Round monetary columns to 2 decimal places
    if col in ("unitprice", "totalprice"):
        df[col] = df[col].round(2)
 
    numeric_issues[col] = {
        "null_count":    int(n_null),
        "negative_count": int(n_negative),
        "zero_count":    int(n_zero),
        "iqr_outliers":  int(n_outliers),
        "iqr_fence":     (round(lower_fence, 2), round(upper_fence, 2)),
    }
 
issues["numeric_validation"] = numeric_issues
 
# ─────────────────────────────────────────────────────────────────────────────
# 8. BUSINESS LOGIC VALIDATION
#    TotalPrice should equal Quantity × UnitPrice (within 1 cent tolerance)
# ─────────────────────────────────────────────────────────────────────────────
log.info("Running business logic checks …")
 
df["_expected_total"] = (df["quantity"] * df["unitprice"]).round(2)
mismatch_mask = ~np.isclose(df["totalprice"].fillna(-1), df["_expected_total"].fillna(-2), rtol=0.01)
n_mismatch = mismatch_mask.sum()
 
if n_mismatch:
    log.warning(f"  {n_mismatch} rows where TotalPrice ≠ Quantity × UnitPrice")
    # Recalculate from source of truth (Qty × UnitPrice)
    df.loc[mismatch_mask, "totalprice"] = df.loc[mismatch_mask, "_expected_total"]
 
# Drop helper column
df.drop(columns=["_expected_total"], inplace=True)
 
issues["business_logic"] = {
    "totalprice_mismatches": int(n_mismatch),
    "action": "Recalculated TotalPrice = Quantity × UnitPrice where mismatched"
}
 
# ─────────────────────────────────────────────────────────────────────────────
# 9. MISSING VALUE STRATEGY
# ─────────────────────────────────────────────────────────────────────────────
log.info("Handling missing values …")
 
missing_strategy = {
    # col           : fill_value  (None = leave as SQL NULL)
    "couponcode"    : "NONE",     # no coupon = explicit 'NONE' string for SQL
    "product"       : None,       # keep NULL – unknown product is bad data
    "paymentmethod" : None,
    "orderstatus"   : None,
    "referralsource": "Unknown",
}
 
missing_report = {}
for col, fill_val in missing_strategy.items():
    if col not in df.columns:
        continue
    n_null = df[col].isna().sum()
    if n_null and fill_val is not None:
        df[col] = df[col].fillna(fill_val)
        log.info(f"  {col}: filled {n_null} NULLs with '{fill_val}'")
    missing_report[col] = {"null_count": int(n_null), "fill_value": fill_val or "SQL NULL"}
 
issues["missing_values"] = missing_report
 
# Report remaining nulls across all columns
final_nulls = df.isnull().sum()
final_nulls = final_nulls[final_nulls > 0]
issues["remaining_nulls_after_cleaning"] = final_nulls.to_dict()
 
# ─────────────────────────────────────────────────────────────────────────────
# 10. ID & CODE FORMAT VALIDATION
# ─────────────────────────────────────────────────────────────────────────────
log.info("Validating ID and code formats …")
 
id_checks = {}
 
# OrderID  → ORD + 6 digits
bad_orderid = ~df["orderid"].str.match(r"^ORD\d{6}$", na=False)
id_checks["orderid"] = {"invalid_format": int(bad_orderid.sum()), "pattern": "ORD######"}
 
# CustomerID → C + 5 digits
bad_custid = ~df["customerid"].str.match(r"^C\d{5}$", na=False)
id_checks["customerid"] = {"invalid_format": int(bad_custid.sum()), "pattern": "C#####"}
 
# TrackingNumber → TRK + 8 digits
bad_trk = ~df["trackingnumber"].str.match(r"^TRK\d{8}$", na=False)
id_checks["trackingnumber"] = {"invalid_format": int(bad_trk.sum()), "pattern": "TRK########"}
 
issues["id_format_checks"] = id_checks
 
for col, info in id_checks.items():
    if info["invalid_format"]:
        log.warning(f"  {col}: {info['invalid_format']} rows do not match pattern {info['pattern']}")
 
# ─────────────────────────────────────────────────────────────────────────────
# 11. FINAL DATA TYPE ENFORCEMENT
#     Cast every column to the correct Python / pandas type so the CSV and
#     SQL Server BULK INSERT do not produce type mismatch errors.
# ─────────────────────────────────────────────────────────────────────────────
log.info("Enforcing final data types …")
 
# Integer columns – use Int64 (nullable integer) to allow NaN without float cast
int_cols = ["quantity", "itemsincart"]
for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
 
# Float columns – standard float64, rounded
float_cols = ["unitprice", "totalprice"]
for col in float_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").round(2)
 
# String / VARCHAR columns – ensure clean str, NaN stays NaN
varchar_cols = [
    "orderid", "customerid", "product", "shippingaddress",
    "paymentmethod", "orderstatus", "trackingnumber",
    "couponcode", "referralsource"
]
for col in varchar_cols:
    if col in df.columns:
        df[col] = df[col].where(df[col].notna(), other=np.nan)
 
# ─────────────────────────────────────────────────────────────────────────────
# 12. COLUMN RE-ORDERING (logical grouping for SQL table)
# ─────────────────────────────────────────────────────────────────────────────
desired_order = [
    "orderid", "date", "customerid",
    "product", "quantity", "unitprice", "totalprice",
    "itemsincart", "couponcode",
    "paymentmethod", "orderstatus",
    "trackingnumber", "shippingaddress", "referralsource"
]
# Only include columns that actually exist
df = df[[c for c in desired_order if c in df.columns]]
 
# ─────────────────────────────────────────────────────────────────────────────
# 13. PRE-EXPORT VALIDATION CHECKS
# ─────────────────────────────────────────────────────────────────────────────
log.info("Running pre-export validation …")
 
validation_results = {}
 
# 13a. No duplicate primary key
validation_results["no_duplicate_orderid"] = bool(df["orderid"].duplicated().sum() == 0)
 
# 13b. All required (NOT NULL) columns populated
not_null_cols = ["orderid", "date", "customerid", "quantity", "unitprice", "totalprice"]
for col in not_null_cols:
    n = df[col].isna().sum()
    validation_results[f"{col}_no_null"] = bool(n == 0)
    if n:
        log.error(f"  VALIDATION FAIL: {col} has {n} NULL values in a NOT NULL column!")
 
# 13c. TotalPrice always positive
validation_results["totalprice_positive"] = bool((df["totalprice"] > 0).all())
 
# 13d. Date values parseable
bad_dates = pd.to_datetime(df["date"], errors="coerce").isna().sum()
validation_results["date_parseable"] = bool(bad_dates == 0)
 
# 13e. Row count integrity
validation_results["row_count_after"] = len(df)
 
issues["pre_export_validation"] = validation_results
all_passed = all(v is True for v in validation_results.values() if isinstance(v, bool))
log.info(f"Pre-export validation: {'ALL PASSED ✓' if all_passed else 'SOME CHECKS FAILED ✗'}")
 
# ─────────────────────────────────────────────────────────────────────────────
# 14. EXPORT CLEANED DATA
# ─────────────────────────────────────────────────────────────────────────────
log.info("Exporting cleaned dataset …")
 
os.makedirs("/mnt/user-data/outputs", exist_ok=True)
 
try:
    # CSV – UTF-8 with BOM for SQL Server BULK INSERT compatibility
    df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
    log.info(f"CSV exported → {OUT_CSV}")
 
    # XLSX – for human review
    df.to_excel(OUT_XLSX, index=False, engine="openpyxl")
    log.info(f"XLSX exported → {OUT_XLSX}")
 
except Exception as e:
    log.error(f"Export failed: {e}")
    raise
 
# ─────────────────────────────────────────────────────────────────────────────
# 15. SQL SERVER SCHEMA RECOMMENDATION
# ─────────────────────────────────────────────────────────────────────────────
 
SQL_SCHEMA = """
-- ============================================================
--  SQL Server Table Schema  (production-ready)
--  Generated by Data Cleaning Pipeline
-- ============================================================
CREATE TABLE dbo.Orders (
    OrderID         NVARCHAR(12)     NOT NULL,   -- ORD######
    [Date]          DATE             NOT NULL,   -- YYYY-MM-DD
    CustomerID      NVARCHAR(8)      NOT NULL,   -- C#####
    Product         NVARCHAR(50)     NULL,
    Quantity        SMALLINT         NOT NULL,   -- 1–5
    UnitPrice       DECIMAL(10, 2)   NOT NULL,
    TotalPrice      DECIMAL(10, 2)   NOT NULL,
    ItemsInCart     SMALLINT         NOT NULL,   -- 1–10
    CouponCode      NVARCHAR(20)     NULL,       -- 'NONE' when no coupon
    PaymentMethod   NVARCHAR(20)     NULL,
    OrderStatus     NVARCHAR(20)     NULL,
    TrackingNumber  NVARCHAR(15)     NULL,       -- TRK########
    ShippingAddress NVARCHAR(200)    NULL,
    ReferralSource  NVARCHAR(30)     NULL,
 
    CONSTRAINT PK_Orders PRIMARY KEY CLUSTERED (OrderID),
 
    -- Business rule constraints
    CONSTRAINT CK_Orders_Quantity    CHECK (Quantity    BETWEEN 1 AND 999),
    CONSTRAINT CK_Orders_UnitPrice   CHECK (UnitPrice   > 0),
    CONSTRAINT CK_Orders_TotalPrice  CHECK (TotalPrice  > 0),
    CONSTRAINT CK_Orders_ItemsInCart CHECK (ItemsInCart BETWEEN 1 AND 999),
    CONSTRAINT CK_Orders_Status      CHECK (OrderStatus IN
        ('Cancelled','Delivered','Pending','Returned','Shipped')),
    CONSTRAINT CK_Orders_Payment     CHECK (PaymentMethod IN
        ('Cash','Credit Card','Debit Card','Gift Card','Online'))
);
 
-- ============================================================
--  BULK INSERT Command  (run AFTER creating the table)
-- ============================================================
BULK INSERT dbo.Orders
FROM 'C:\\data\\orders_cleaned.csv'
WITH (
    FIRSTROW        = 2,
    FIELDTERMINATOR = ',',
    ROWTERMINATOR   = '\\n',
    CODEPAGE        = '65001',   -- UTF-8
    TABLOCK
);
 
-- ============================================================
--  Recommended Indexes
-- ============================================================
CREATE NONCLUSTERED INDEX IX_Orders_CustomerID   ON dbo.Orders (CustomerID);
CREATE NONCLUSTERED INDEX IX_Orders_Date         ON dbo.Orders ([Date]);
CREATE NONCLUSTERED INDEX IX_Orders_OrderStatus  ON dbo.Orders (OrderStatus);
"""
 
# ─────────────────────────────────────────────────────────────────────────────
# 16. GENERATE FINAL REPORT
# ─────────────────────────────────────────────────────────────────────────────
 
def build_report() -> str:
    lines = []
    sep   = "=" * 70
 
    lines += [sep, "  DATA CLEANING PIPELINE — FINAL REPORT", sep, ""]
    lines += [f"  Run timestamp : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"]
    lines += [f"  Input file    : {INPUT_FILE}"]
    lines += [f"  Output CSV    : {OUT_CSV}"]
    lines += [f"  Output XLSX   : {OUT_XLSX}", ""]
 
    lines += [sep, "  1. DATASET SHAPE", sep]
    lines += [f"  Original  : {original_shape[0]:,} rows × {original_shape[1]} columns"]
    lines += [f"  Cleaned   : {df.shape[0]:,} rows × {df.shape[1]} columns", ""]
 
    lines += [sep, "  2. COLUMN NAMES RENAMED", sep]
    renamed = issues["column_rename"]["renamed"]
    if renamed:
        for orig, new in renamed.items():
            lines.append(f"  '{orig}'  →  '{new}'")
    else:
        lines.append("  None — all names were already clean")
    lines.append("")
 
    lines += [sep, "  3. DUPLICATE ANALYSIS", sep]
    d = issues["duplicates"]
    lines += [
        f"  Full duplicate rows    : {d['full_duplicate_rows']}",
        f"  Duplicate OrderIDs     : {d['duplicate_orderids']}",
        f"  Action                 : {d['action']}", ""
    ]
 
    lines += [sep, "  4. STRING COLUMN CLEANING", sep]
    sc = issues["string_cleaning"]["columns_with_changes"]
    if sc:
        for col, n in sc.items():
            lines.append(f"  {col}: {n} cells modified")
    else:
        lines.append("  No whitespace / encoding issues found")
    lines.append("")
 
    lines += [sep, "  5. CATEGORICAL STANDARDISATION", sep]
    cat = issues["categorical_standardisation"]
    if isinstance(cat, dict) and cat:
        for col, info in cat.items():
            lines.append(f"  {col}: {info}")
    else:
        lines.append("  All categorical values within expected sets ✓")
    lines.append("")
 
    lines += [sep, "  6. DATE COLUMNS", sep]
    for col, info in issues["date_columns"].items():
        lines += [
            f"  Column: {col}",
            f"    Dtype before    : {info['dtype_before']}",
            f"    Null after parse: {info['null_after_parse']}",
            f"    Future dates    : {info['future_dates']}",
            f"    Dates < 2000    : {info['dates_before_2000']}", ""
        ]
 
    lines += [sep, "  7. NUMERIC VALIDATION & OUTLIERS", sep]
    for col, info in issues["numeric_validation"].items():
        lines += [
            f"  Column: {col}",
            f"    Nulls           : {info['null_count']}",
            f"    Negatives       : {info['negative_count']}",
            f"    Zeros           : {info['zero_count']}",
            f"    IQR outliers    : {info['iqr_outliers']}  fence={info['iqr_fence']}", ""
        ]
 
    lines += [sep, "  8. BUSINESS LOGIC", sep]
    bl = issues["business_logic"]
    lines += [
        f"  TotalPrice mismatches : {bl['totalprice_mismatches']}",
        f"  Action                : {bl['action']}", ""
    ]
 
    lines += [sep, "  9. MISSING VALUE HANDLING", sep]
    for col, info in issues["missing_values"].items():
        lines.append(f"  {col}: {info['null_count']} NULLs → filled with '{info['fill_value']}'")
    remaining = issues.get("remaining_nulls_after_cleaning", {})
    if remaining:
        lines += ["", "  Remaining NULLs (intentional SQL NULLs):"]
        for col, n in remaining.items():
            lines.append(f"    {col}: {n}")
    lines.append("")
 
    lines += [sep, "  10. ID FORMAT VALIDATION", sep]
    for col, info in issues["id_format_checks"].items():
        status = "✓" if info["invalid_format"] == 0 else "✗"
        lines.append(f"  {status} {col}: {info['invalid_format']} non-conforming values (pattern: {info['pattern']})")
    lines.append("")
 
    lines += [sep, "  11. PRE-EXPORT VALIDATION RESULTS", sep]
    for check, result in issues["pre_export_validation"].items():
        status = "PASS ✓" if result is True else "INFO" if isinstance(result, int) else "FAIL ✗"
        lines.append(f"  [{status}] {check}: {result}")
    lines.append("")
 
    lines += [sep, "  12. SQL SERVER RECOMMENDED DATA TYPES", sep]
    sql_types = {
        "orderid":         "NVARCHAR(12)  NOT NULL  [PK]",
        "date":            "DATE          NOT NULL",
        "customerid":      "NVARCHAR(8)   NOT NULL",
        "product":         "NVARCHAR(50)  NULL",
        "quantity":        "SMALLINT      NOT NULL",
        "unitprice":       "DECIMAL(10,2) NOT NULL",
        "totalprice":      "DECIMAL(10,2) NOT NULL",
        "itemsincart":     "SMALLINT      NOT NULL",
        "couponcode":      "NVARCHAR(20)  NULL",
        "paymentmethod":   "NVARCHAR(20)  NULL",
        "orderstatus":     "NVARCHAR(20)  NULL",
        "trackingnumber":  "NVARCHAR(15)  NULL",
        "shippingaddress": "NVARCHAR(200) NULL",
        "referralsource":  "NVARCHAR(30)  NULL",
    }
    for col, dtype in sql_types.items():
        lines.append(f"  {col:<20} {dtype}")
    lines.append("")
 
    lines += [sep, "  13. POTENTIAL SQL SERVER IMPORT RISKS DETECTED", sep]
    lines += [
        "  ✓ CouponCode NULLs — handled: filled with 'NONE' string",
        "  ✓ TotalPrice IQR outliers (8 rows) — values are within absolute",
        "    business bounds, no action taken; flag for business review",
        "  ✓ Date column exported as ISO 8601 string (YYYY-MM-DD)",
        "  ✓ CSV encoded UTF-8 with BOM (CODEPAGE 65001) for BULK INSERT",
        "  ✓ Integer cols use pandas Int64 to preserve NULL without float cast",
        "  ✓ No special characters in column names", ""
    ]
 
    lines += [sep, "  14. SQL SCHEMA & BULK INSERT COMMAND", sep]
    lines += [SQL_SCHEMA]
 
    return "\n".join(lines)
 
 
report_text = build_report()
print(report_text)
 
with open(REPORT_FILE, "w", encoding="utf-8") as f:
    f.write(report_text)
 
log.info(f"Report written → {REPORT_FILE}")
log.info("Pipeline complete ✓")

2026-05-23 10:15:19,045 [INFO] Cleaning string columns …
2026-05-23 10:15:19,100 [INFO] Standardising categorical columns …
2026-05-23 10:15:19,107 [INFO] Cleaning date columns …
2026-05-23 10:15:19,115 [INFO] Validating numeric columns …
2026-05-23 10:15:19,127 [INFO] Running business logic checks …
2026-05-23 10:15:19,134 [INFO] Handling missing values …
2026-05-23 10:15:19,136 [INFO]   couponcode: filled 309 NULLs with 'NONE'
2026-05-23 10:15:19,140 [INFO] Validating ID and code formats …
2026-05-23 10:15:19,146 [INFO] Enforcing final data types …
2026-05-23 10:15:19,154 [INFO] Running pre-export validation …
2026-05-23 10:15:19,162 [INFO] Pre-export validation: ALL PASSED ✓
--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\Access\anaconda3\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Access\anaconda3\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.

  DATA CLEANING PIPELINE — FINAL REPORT

  Run timestamp : 2026-05-23 10:15:19
  Input file    : C:\Users\Access\Downloads\New folder\Dataset for Data Analytics.xlsx
  Output CSV    : C:\Users\Access\Downloads\New folder\orders_cleaned.csv
  Output XLSX   : /mnt/user-data/outputs/orders_cleaned.xlsx

  1. DATASET SHAPE
  Original  : 1,200 rows × 14 columns
  Cleaned   : 1,200 rows × 14 columns

  2. COLUMN NAMES RENAMED
  'OrderID'  →  'orderid'
  'Date'  →  'date'
  'CustomerID'  →  'customerid'
  'Product'  →  'product'
  'Quantity'  →  'quantity'
  'UnitPrice'  →  'unitprice'
  'ShippingAddress'  →  'shippingaddress'
  'PaymentMethod'  →  'paymentmethod'
  'OrderStatus'  →  'orderstatus'
  'TrackingNumber'  →  'trackingnumber'
  'ItemsInCart'  →  'itemsincart'
  'CouponCode'  →  'couponcode'
  'ReferralSource'  →  'referralsource'
  'TotalPrice'  →  'totalprice'

  3. DUPLICATE ANALYSIS
  Full duplicate rows    : 0
  Duplicate OrderIDs     : 0
  Action                 : Removed – la

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\Access\anaconda3\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Access\anaconda3\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2713' in position 49: character maps to <undefined>
Call stack:
  File "C:\Users\Access\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\Access\anaconda3\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\Access\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Access\anaconda3\Lib\site-packages\traitlets\config\application.py", l